# Anchor UI Evals — Runs Dashboard

This notebook visualizes `logs/runs.jsonl` (JSONL trace events written by `trace_logger.py`).

If you don’t have plotting deps installed yet, you can still run the summary tables.

In [1]:
from __future__ import annotations

import json
import os
from datetime import datetime
from pathlib import Path
from typing import Any, Optional


def find_log_path() -> Path:
    """Find logs/runs.jsonl reliably across VS Code + Colab CWDs."""
    env_path = os.getenv("EVAL_LOG_PATH")
    if env_path:
        p = Path(env_path).expanduser()
        if p.exists():
            return p.resolve()

    candidates = [
        Path("logs") / "runs.jsonl",  # if CWD == backend/evals
        Path("backend/evals/logs/runs.jsonl"),  # if CWD == repo root
        Path("anchor_ui/backend/evals/logs/runs.jsonl"),  # if CWD above repo
    ]
    for c in candidates:
        p = c.resolve()
        if p.exists():
            return p

    # Colab often runs with CWD=/content; search for the repo checkout.
    matches = list(Path.cwd().glob("**/backend/evals/logs/runs.jsonl"))
    if matches:
        return matches[0].resolve()
    matches = list(Path.cwd().glob("**/logs/runs.jsonl"))
    if matches:
        return matches[0].resolve()

    # Default (will fail with a helpful message in load_jsonl).
    return Path("backend/evals/logs/runs.jsonl").resolve()


LOG_PATH = find_log_path()
print("CWD:", Path.cwd())
print("LOG_PATH:", LOG_PATH)
LOG_PATH

CWD: c:\Users\lamjat\Documents\Novia_Work_Projects\2025\KB\doc_kb-ui-v2\anchor_ui\backend\evals
LOG_PATH: C:\Users\lamjat\Documents\Novia_Work_Projects\2025\KB\doc_kb-ui-v2\anchor_ui\backend\evals\logs\runs.jsonl


WindowsPath('C:/Users/lamjat/Documents/Novia_Work_Projects/2025/KB/doc_kb-ui-v2/anchor_ui/backend/evals/logs/runs.jsonl')

In [2]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(
            "Log file not found: "
            + str(path.resolve())
            + "\n\n"
            + "Tips:\n"
            + "- If using VS Code Colab, your CWD is often /content.\n"
            + "- Set env var EVAL_LOG_PATH to the full path of runs.jsonl.\n"
            + "- Or run a quick search cell: !find /content -path '*backend/evals/logs/runs.jsonl' 2>/dev/null | head\n"
        )
    events: list[dict[str, Any]] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return events


events = load_jsonl(LOG_PATH)
len(events)

131

In [3]:
# Optional: use pandas if available (recommended)
try:
    import pandas as pd
except Exception as e:
    pd = None
    print("pandas not available; install with: pip install pandas")
    print(e)


def parse_ts(ts: str) -> Optional[datetime]:
    try:
        # ts looks like: 2025-12-11T12:34:56.123Z
        return datetime.fromisoformat(ts.replace("Z", "+00:00"))
    except Exception:
        return None


if pd is not None:
    df = pd.DataFrame(events)
    if "ts" in df.columns:
        df["ts"] = df["ts"].map(parse_ts)
    df.head(5)
else:
    events[:3]

## Pick a run

Each event has a `run_id`. By default, the app uses `EVAL_RUN_ID` or generates one.

In [4]:
if pd is None:
    run_ids = sorted({e.get("run_id") for e in events if e.get("run_id")})
    run_ids[:10], len(run_ids)
else:
    run_ids = (
        df["run_id"].dropna().astype(str).drop_duplicates().sort_values().tolist()
        if "run_id" in df.columns
        else []
    )
    run_ids[:10], len(run_ids)

In [5]:
# Choose a run id.
# Tip: keep EVAL_RUN_ID fixed per experiment so charts stay clean.

RUN_ID = run_ids[-1] if run_ids else None
RUN_ID

'6b67fc3c1447403f929173ad7da8da7f'

In [6]:
if RUN_ID is None:
    raise RuntimeError("No run_id found in events")


if pd is None:
    run_events = [e for e in events if e.get("run_id") == RUN_ID]
    print("events:", len(run_events))
    by_type = {}
    for e in run_events:
        by_type[e.get("type") or "<missing>"] = by_type.get(e.get("type") or "<missing>", 0) + 1
    by_type
else:
    run_df = df[df["run_id"].astype(str) == str(RUN_ID)].copy()
    display(run_df[[c for c in ["ts", "type", "run_id"] if c in run_df.columns]].head(10))
    run_df["type"].value_counts(dropna=False)

,ts,type,run_id
6,2025-12-15 11:06:29.573662+00:00,llm_call,6b67fc3c1447403f929173ad7da8da7f
7,2025-12-15 11:06:30.054444+00:00,embedding,6b67fc3c1447403f929173ad7da8da7f
8,2025-12-15 11:06:31.504638+00:00,retrieval,6b67fc3c1447403f929173ad7da8da7f
9,2025-12-15 11:06:31.505400+00:00,state,6b67fc3c1447403f929173ad7da8da7f
10,2025-12-15 11:06:38.947089+00:00,llm_call,6b67fc3c1447403f929173ad7da8da7f
11,2025-12-15 11:06:39.158024+00:00,embedding,6b67fc3c1447403f929173ad7da8da7f
12,2025-12-15 11:06:39.238633+00:00,retrieval,6b67fc3c1447403f929173ad7da8da7f
13,2025-12-15 11:06:39.238633+00:00,state,6b67fc3c1447403f929173ad7da8da7f
14,2025-12-15 11:06:45.509159+00:00,llm_call,6b67fc3c1447403f929173ad7da8da7f
15,2025-12-15 11:06:45.731350+00:00,embedding,6b67fc3c1447403f929173ad7da8da7f


## Summary tables

In [7]:
if pd is None:
    print("Install pandas for better tables: pip install pandas")
else:
    def summarize_numeric(frame: "pd.DataFrame", cols: list[str]) -> "pd.DataFrame":
        cols = [c for c in cols if c in frame.columns]
        if not cols:
            return pd.DataFrame()
        return frame[cols].describe(percentiles=[0.5, 0.95, 0.99]).T

    llm = run_df[run_df["type"] == "llm_call"] if "type" in run_df.columns else run_df.iloc[0:0]
    ret = run_df[run_df["type"] == "retrieval"] if "type" in run_df.columns else run_df.iloc[0:0]
    emb = run_df[run_df["type"] == "embedding"] if "type" in run_df.columns else run_df.iloc[0:0]

    print("LLM calls:")
    display(summarize_numeric(llm, ["latency_ms", "prompt_tokens_est"]))
    if "usage" in llm.columns:
        # usage may be a dict; normalize if present.
        usage_df = pd.json_normalize(llm["usage"].dropna())
        usage_df.columns = [f"usage.{c}" for c in usage_df.columns]
        display(usage_df.describe(percentiles=[0.5, 0.95, 0.99]).T)

    print("Retrieval:")
    display(summarize_numeric(ret, ["latency_ms", "top_k", "result_count", "total_chunk_tokens"]))

    print("Embeddings:")
    display(summarize_numeric(emb, ["latency_ms", "batch_size", "input_tokens_est"]))


LLM calls:


,count,mean,std,min,50%,95%,99%,max
latency_ms,35.0,13612.149674,6811.260251,2757.7379,15193.7848,22208.87043,28712.44416,31943.5648
prompt_tokens_est,35.0,0.000000,0.000000,0.0000,0.0000,0.00000,0.00000,0.0000


ValueError: Cannot describe a DataFrame without columns

## Charts

Uses Plotly if installed (recommended), otherwise Matplotlib.

In [14]:
if pd is None:
    raise RuntimeError("Install pandas to enable plotting: pip install pandas")

try:
    import plotly.express as px
    PLOTLY = True
except Exception:
    PLOTLY = False

PLOTLY

True

In [15]:
plot_df = run_df.copy()
plot_df = plot_df.dropna(subset=["ts"]) if "ts" in plot_df.columns else plot_df
plot_df = plot_df.sort_values("ts") if "ts" in plot_df.columns else plot_df

if PLOTLY:
    fig = px.scatter(
        plot_df,
        x="ts",
        y="latency_ms",
        color="type",
        title=f"Latency over time (run_id={RUN_ID})",
        hover_data=[c for c in ["top_k", "result_count", "prompt_tokens_est"] if c in plot_df.columns],
    )
    fig.update_traces(mode="markers")
    fig.show()
else:
    import matplotlib.pyplot as plt

    if "ts" not in plot_df.columns or "latency_ms" not in plot_df.columns:
        raise RuntimeError("Missing ts/latency_ms columns")

    plt.figure(figsize=(12, 5))
    for t, g in plot_df.dropna(subset=["type"]).groupby("type"):
        if "latency_ms" in g.columns:
            plt.plot(g["ts"], g["latency_ms"], marker="o", linestyle="-", label=str(t))
    plt.title(f"Latency over time (run_id={RUN_ID})")
    plt.xlabel("time")
    plt.ylabel("latency_ms")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [16]:
if PLOTLY:
    fig = px.histogram(
        run_df,
        x="latency_ms",
        color="type",
        barmode="overlay",
        nbins=40,
        title=f"Latency distribution (run_id={RUN_ID})",
    )
    fig.update_traces(opacity=0.6)
    fig.show()
else:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 5))
    for t, g in run_df.dropna(subset=["type"]).groupby("type"):
        if "latency_ms" in g.columns:
            plt.hist(g["latency_ms"].dropna(), bins=40, alpha=0.5, label=str(t))
    plt.title(f"Latency distribution (run_id={RUN_ID})")
    plt.xlabel("latency_ms")
    plt.ylabel("count")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Next improvements (implemented)

- Per-model/provider breakdowns (from `llm_call` / `embedding` events).
- Retrieval quality proxies (fill rate, doc diversity, overlap).
- Export a shareable HTML report (tables + optional Plotly charts).

In [17]:
# Per-model/provider breakdowns
if pd is None:
    raise RuntimeError("Install pandas to run this section: pip install pandas")


def q95(s: "pd.Series") -> float:
    s = s.dropna()
    return float(s.quantile(0.95)) if len(s) else 0.0


llm_df = run_df[run_df.get("type") == "llm_call"].copy() if "type" in run_df.columns else run_df.iloc[0:0]
emb_df = run_df[run_df.get("type") == "embedding"].copy() if "type" in run_df.columns else run_df.iloc[0:0]

for frame in (llm_df, emb_df):
    for c in ["provider", "model"]:
        if c not in frame.columns:
            frame[c] = "<missing>"

print("LLM calls by provider/model:")
if llm_df.empty:
    print("No llm_call events found for this run")
else:
    llm_group = (
        llm_df.groupby(["provider", "model"], dropna=False)
        .agg(
            calls=("type", "size"),
            latency_ms_mean=("latency_ms", "mean"),
            latency_ms_p95=("latency_ms", q95),
            prompt_tokens_est_mean=("prompt_tokens_est", "mean"),
        )
        .sort_values(["calls", "latency_ms_mean"], ascending=[False, False])
    )
    display(llm_group)

print("Embeddings by provider/model:")
if emb_df.empty:
    print("No embedding events found for this run")
else:
    emb_group = (
        emb_df.groupby(["provider", "model"], dropna=False)
        .agg(
            calls=("type", "size"),
            latency_ms_mean=("latency_ms", "mean"),
            latency_ms_p95=("latency_ms", q95),
            input_tokens_est_mean=("input_tokens_est", "mean"),
        )
        .sort_values(["calls", "latency_ms_mean"], ascending=[False, False])
    )
    display(emb_group)

if PLOTLY and not llm_df.empty:
    llm_plot = llm_group.reset_index()
    llm_plot["label"] = llm_plot["provider"].astype(str) + " / " + llm_plot["model"].astype(str)
    fig = px.bar(
        llm_plot,
        x="label",
        y="latency_ms_p95",
        title=f"LLM latency p95 by provider/model (run_id={RUN_ID})",
    )
    fig.update_layout(xaxis_title="provider / model", yaxis_title="latency_ms p95")
    fig.show()

LLM calls by provider/model:


,,calls,latency_ms_mean,latency_ms_p95,prompt_tokens_est_mean
provider,model,,,,
"AzureProvider(name=azure, base_url=https://oai-new-fmi-sim-test-swe.openai.azure.com/openai/)",novia-gpt-5-nano,35,13612.149674,22208.87043,0.0


Embeddings by provider/model:


,,calls,latency_ms_mean,latency_ms_p95,input_tokens_est_mean
provider,model,,,,
azure,azure-text-embedding-3-large,30,211.902553,494.99562,17.866667


In [18]:
# Retrieval quality proxies (until you have ground-truth hit-rate labels)
if pd is None:
    raise RuntimeError("Install pandas to run this section: pip install pandas")

ret_df = run_df[run_df.get("type") == "retrieval"].copy() if "type" in run_df.columns else run_df.iloc[0:0]
if ret_df.empty:
    print("No retrieval events found for this run")
else:
    if "top_k" in ret_df.columns and "result_count" in ret_df.columns:
        ret_df["fill_rate"] = ret_df["result_count"].fillna(0) / ret_df["top_k"].replace({0: None})

    def doc_diversity(x: Any) -> int:
        if isinstance(x, list):
            return len({d for d in x if d})
        return 0

    if "doc_ids" in ret_df.columns:
        ret_df["doc_diversity"] = ret_df["doc_ids"].map(doc_diversity)

    def jaccard(a: Any, b: Any) -> float:
        sa = {x for x in (a or []) if x} if isinstance(a, list) else set()
        sb = {x for x in (b or []) if x} if isinstance(b, list) else set()
        if not sa and not sb:
            return 0.0
        return len(sa & sb) / len(sa | sb)

    if "ts" in ret_df.columns:
        ret_df = ret_df.sort_values("ts")
    if "doc_ids" in ret_df.columns:
        ret_df["prev_doc_overlap"] = [
            0.0
        ] + [
            jaccard(ret_df.iloc[i - 1]["doc_ids"], ret_df.iloc[i]["doc_ids"])
            for i in range(1, len(ret_df))
        ]

    cols = [c for c in ["ts", "top_k", "result_count", "fill_rate", "doc_diversity", "total_chunk_tokens", "latency_ms", "prev_doc_overlap"] if c in ret_df.columns]
    display(ret_df[cols].head(30))
    display(ret_df[[c for c in cols if c != "ts"]].describe(percentiles=[0.5, 0.95, 0.99]).T)

    if PLOTLY and "ts" in ret_df.columns and "fill_rate" in ret_df.columns:
        fig = px.line(ret_df, x="ts", y="fill_rate", title=f"Retrieval fill rate over time (run_id={RUN_ID})")
        fig.update_layout(yaxis_title="result_count / top_k")
        fig.show()
    if PLOTLY and "ts" in ret_df.columns and "doc_diversity" in ret_df.columns:
        fig = px.line(ret_df, x="ts", y="doc_diversity", title=f"Document diversity over time (run_id={RUN_ID})")
        fig.update_layout(yaxis_title="unique documents in results")
        fig.show()

,ts,top_k,result_count,fill_rate,doc_diversity,total_chunk_tokens,latency_ms,prev_doc_overlap
8,2025-12-15 11:06:31.504638+00:00,5.0,5.0,1.0,1,1824.0,1797.324181,0.0
12,2025-12-15 11:06:39.238633+00:00,5.0,5.0,1.0,1,2213.0,286.458015,1.0
16,2025-12-15 11:06:45.814287+00:00,5.0,5.0,1.0,1,2213.0,298.946857,1.0
22,2025-12-15 11:07:44.782925+00:00,5.0,5.0,1.0,1,1981.0,340.475321,1.0
26,2025-12-15 11:08:07.524316+00:00,5.0,5.0,1.0,1,2233.0,294.673443,1.0
30,2025-12-15 11:08:14.486016+00:00,5.0,5.0,1.0,1,2419.0,194.852352,1.0
34,2025-12-15 11:08:26.184447+00:00,5.0,5.0,1.0,1,2419.0,229.850769,1.0
38,2025-12-15 11:08:43.641646+00:00,5.0,5.0,1.0,1,2419.0,279.673100,1.0
42,2025-12-15 11:08:50.613449+00:00,5.0,5.0,1.0,1,2419.0,186.865330,1.0
46,2025-12-15 11:09:09.413382+00:00,5.0,5.0,1.0,1,2419.0,1035.903215,1.0


,count,mean,std,min,50%,95%,99%,max
top_k,30.0,5.000000,0.000000,5.000000,5.000000,5.000000,5.000000,5.000000
result_count,30.0,5.000000,0.000000,5.000000,5.000000,5.000000,5.000000,5.000000
fill_rate,30.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
doc_diversity,30.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
total_chunk_tokens,30.0,2364.033333,140.779966,1824.000000,2419.000000,2419.000000,2419.000000,2419.000000
latency_ms,30.0,367.990462,319.504530,158.361197,278.057694,842.332911,1576.512101,1797.324181
prev_doc_overlap,30.0,0.966667,0.182574,0.000000,1.000000,1.000000,1.000000,1.000000


In [13]:
# Export a shareable HTML report (writes into backend/evals/reports/)
if pd is None:
    raise RuntimeError("Install pandas to export HTML reports: pip install pandas")

import html

REPORT_DIR = LOG_PATH.parent.parent / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = REPORT_DIR / f"run_report_{RUN_ID}.html"


def df_to_html(frame: "pd.DataFrame", title: str) -> str:
    if frame is None or frame.empty:
        return f"<h2>{html.escape(title)}</h2><p><em>none</em></p>"
    return (
        f"<h2>{html.escape(title)}</h2>"
        + frame.to_html(index=False, escape=True, border=0, classes="tbl")
    )


parts: list[str] = []
parts.append("<!doctype html><html><head><meta charset='utf-8'>")
parts.append(f"<title>Anchor UI Eval Report — {html.escape(str(RUN_ID))}</title>")
parts.append(
    """
    <style>
      body{font-family:ui-sans-serif,system-ui,-apple-system,Segoe UI,Roboto,Arial; margin:24px; color:#0f172a;}
      h1{margin:0 0 8px 0; font-size:22px;}
      .meta{color:#475569; margin-bottom:16px;}
      .grid{display:grid; grid-template-columns:1fr; gap:16px;}
      .card{border:1px solid #e2e8f0; border-radius:12px; padding:16px; background:#ffffff;}
      .tbl{border-collapse:collapse; width:100%; font-size:12px;}
      .tbl th,.tbl td{border-bottom:1px solid #e2e8f0; padding:8px; text-align:left; vertical-align:top;}
      .tbl th{background:#f8fafc; position:sticky; top:0;}
      code{background:#f1f5f9; padding:2px 6px; border-radius:6px;}
    </style>
    """
)
parts.append("</head><body>")
parts.append(f"<h1>Anchor UI Eval Report</h1>")
time_range = ""
if "ts" in run_df.columns and run_df["ts"].notna().any():
    tmin = run_df["ts"].min()
    tmax = run_df["ts"].max()
    time_range = f"{tmin} → {tmax}"
parts.append(
    f"<div class='meta'>run_id=<code>{html.escape(str(RUN_ID))}</code> &nbsp; events=<code>{len(run_df)}</code> &nbsp; {html.escape(time_range)}</div>"
)
parts.append("<div class='grid'>")

# Tables
parts.append("<div class='card'>" + df_to_html(run_df[[c for c in ["ts","type","latency_ms","provider","model","top_k","result_count"] if c in run_df.columns]].head(50), "Sample events (first 50)") + "</div>")

if "type" in run_df.columns:
    llm_df = run_df[run_df["type"] == "llm_call"].copy()
    ret_df = run_df[run_df["type"] == "retrieval"].copy()
    emb_df = run_df[run_df["type"] == "embedding"].copy()
else:
    llm_df = run_df.iloc[0:0]
    ret_df = run_df.iloc[0:0]
    emb_df = run_df.iloc[0:0]

def q95(s: "pd.Series") -> float:
    s = s.dropna()
    return float(s.quantile(0.95)) if len(s) else 0.0

if not llm_df.empty:
    for c in ["provider", "model"]:
        if c not in llm_df.columns:
            llm_df[c] = "<missing>"
    llm_group = (
        llm_df.groupby(["provider", "model"], dropna=False)
        .agg(
            calls=("type", "size"),
            latency_ms_mean=("latency_ms", "mean"),
            latency_ms_p95=("latency_ms", q95),
            prompt_tokens_est_mean=("prompt_tokens_est", "mean"),
        )
        .reset_index()
        .sort_values(["calls", "latency_ms_mean"], ascending=[False, False])
    )
    parts.append("<div class='card'>" + df_to_html(llm_group, "LLM calls by provider/model") + "</div>")

if not emb_df.empty:
    for c in ["provider", "model"]:
        if c not in emb_df.columns:
            emb_df[c] = "<missing>"
    emb_group = (
        emb_df.groupby(["provider", "model"], dropna=False)
        .agg(
            calls=("type", "size"),
            latency_ms_mean=("latency_ms", "mean"),
            latency_ms_p95=("latency_ms", q95),
            input_tokens_est_mean=("input_tokens_est", "mean"),
        )
        .reset_index()
        .sort_values(["calls", "latency_ms_mean"], ascending=[False, False])
    )
    parts.append("<div class='card'>" + df_to_html(emb_group, "Embeddings by provider/model") + "</div>")

if not ret_df.empty and "top_k" in ret_df.columns and "result_count" in ret_df.columns:
    ret_group = ret_df.copy()
    ret_group["fill_rate"] = ret_group["result_count"].fillna(0) / ret_group["top_k"].replace({0: None})
    ret_summary = (
        ret_group[[c for c in ["latency_ms", "top_k", "result_count", "fill_rate", "total_chunk_tokens"] if c in ret_group.columns]]
        .describe(percentiles=[0.5, 0.95, 0.99])
        .reset_index()
        .rename(columns={"index": "stat"})
    )
    parts.append("<div class='card'>" + df_to_html(ret_summary, "Retrieval summary") + "</div>")

# Charts (optional)
if PLOTLY:
    import plotly.io as pio

    if "ts" in run_df.columns and "latency_ms" in run_df.columns:
        fig = px.scatter(
            run_df.dropna(subset=["ts"]).sort_values("ts"),
            x="ts",
            y="latency_ms",
            color="type" if "type" in run_df.columns else None,
            title="Latency over time",
        )
        parts.append("<div class='card'>" + pio.to_html(fig, include_plotlyjs='cdn', full_html=False) + "</div>")
    if "latency_ms" in run_df.columns and "type" in run_df.columns:
        fig = px.histogram(run_df, x="latency_ms", color="type", barmode="overlay", nbins=40, title="Latency distribution")
        fig.update_traces(opacity=0.6)
        parts.append("<div class='card'>" + pio.to_html(fig, include_plotlyjs=False, full_html=False) + "</div>")

parts.append("</div></body></html>")
OUT_PATH.write_text("\n".join(parts), encoding="utf-8")
print("Wrote:", OUT_PATH)


Wrote: C:\Users\lamjat\Documents\Novia_Work_Projects\2025\KB\doc_kb-ui-v2\anchor_ui\backend\evals\reports\run_report_6b67fc3c1447403f929173ad7da8da7f.html
